# Hyperparameter Tuning
The purpose of this notebook is to improve our baseline ML models, tuning their hyperparameters and selecting the strongest model based on evidence. Both the Logistic Regression and Random Forest models will be tuned and then the best model will be saved for future notebooks.

In [0]:
import joblib
import numpy as np
from pathlib import Path
from scipy.sparse import issparse

# Get this notebook’s Workspace path, then jump to repo root and into /artifacts
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_ws_path = ctx.notebookPath().get()                    # like: /Users/.../csai382.../notebooks/csai_lab_5_3
repo_ws_path = nb_ws_path.rsplit("/notebooks/", 1)[0]   # like: /Users/.../csai382...
ART = Path("/Workspace" + repo_ws_path) / "artifacts"

# Load artifacts (match your screenshot)
pipeline = joblib.load(ART / "stedi_feature_pipeline.pkl")

X_train_transformed = np.load(ART / "X_train_transformed.npy", allow_pickle=True)
X_test_transformed  = np.load(ART / "X_test_transformed.npy", allow_pickle=True)

y_train = joblib.load(ART / "y_train.pkl")
y_test  = joblib.load(ART / "y_test.pkl")

def to_float_matrix(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 0:
        arr = arr.item()
        if issparse(arr):
            arr = arr.toarray()
        arr = np.array(arr, dtype=float)
    elif arr.dtype == object:
        arr = np.array([
            x.toarray() if issparse(x) else np.array(x, dtype=float)
            for x in arr
        ])
        arr = np.vstack(arr)
    elif issparse(arr):
        arr = arr.toarray()
    else:
        arr = np.array(arr, dtype=float)
    return arr

X_train = to_float_matrix(X_train_transformed)
X_test  = to_float_matrix(X_test_transformed)

y_train = np.ravel(y_train)
y_test  = np.ravel(y_test)

X_train.shape, X_test.shape, y_train.shape, y_test.shape


In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

log_reg_params = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear"]
}

log_reg_grid = GridSearchCV(
    LogisticRegression(max_iter=300),
    log_reg_params,
    cv=3,
    scoring="accuracy"
)

log_reg_grid.fit(X_train, y_train)

log_reg_best_params = log_reg_grid.best_params_
log_reg_best_score = log_reg_grid.best_score_

log_reg_best_params, log_reg_best_score


# Logistic Regression Parameters

Best parameters: {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}

Best score: 0.9511214840660257

In [0]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(),
    rf_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

rf_best_params = rf_grid.best_params_
rf_best_score = rf_grid.best_score_

rf_best_params, rf_best_score


# Random Forest Parameters
Best Parameters: {'max_depth': 5,
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 50}

Best Score: 0.9511214840660257

In [0]:
results = {
    "Logistic Regression (tuned)": log_reg_best_score,
    "Random Forest (tuned)": rf_best_score
}
results

Both Logistic Regression and Random Forest have identical scores upon tuning; due to this, and the fact that Random Forest takes 10+ times as long to run than Logistic Regression, without any real benefits, we are going to default to Logistic Regression for the additional speed that we gain.

In [0]:
# Choose the better model based on best_score_
if rf_best_score > log_reg_best_score:
    best_model = rf_grid.best_estimator_
    best_model_name = "Random Forest"
else:
    best_model = log_reg_grid.best_estimator_
    best_model_name = "Logistic Regression"
forest_model = rf_grid.best_estimator_
best_model_name, best_model

In [0]:
import joblib
from pathlib import Path

# Find this repo's /artifacts folder from the notebook path
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_ws_path = ctx.notebookPath().get()
repo_ws_path = nb_ws_path.rsplit("/notebooks/", 1)[0]
ART = Path("/Workspace" + repo_ws_path) / "artifacts"
ART.mkdir(parents=True, exist_ok=True)

# Save into repo artifacts
joblib.dump(best_model, str(ART / "stedi_best_model.pkl"))
joblib.dump(forest_model, str(ART / "stedi_forest_model.pkl"))

# Model Evaluation and Ethics Reflection
According to our tests after tuning each of the models, we find that both result in identical test scores. Because of this, we default to which one performed faster, which was Logistic Regression. We scored specifically on accuracy, resulting in identical scores. In markdown cells above this you will find the best resulting parameters for accuracy for each model.

I was surprised to find that both models were just as accurate; I had assumed that the faster one sacrificed accuracy for speed, but it seems that is not the case. If I had more time, I'd be trying out a variety of models, increased data, random values in each of the parameters, even scoring based on other criteria instead of accuracy.

I scored based on accuracy, as mentioned before. One thing to be aware of is how scoring entirely on accuracy could result in some kind of bias. Hence, if I clearly state what I am scoring on, I exhibit proper transparency for what we are doing.

Repentance in the Gospel requires we look at what we have done from God's perspective, decide what we did was wrong and why, and then turn to God for help to stop doing such actions, thought patterns, etc. in an attempt to become better and more like Him.

In [0]:
import numpy as np
from collections import Counter

print("y_train:", Counter(y_train))
print("y_test :", Counter(y_test))

# sanity check: what does the model think probabilities look like?
if hasattr(best_model, "predict_proba"):
    proba = best_model.predict_proba(X_test)
    # find index of "no_step" if labels are strings
    print("classes:", best_model.classes_)
    print("proba summary (first class):", np.min(proba[:,0]), np.mean(proba[:,0]), np.max(proba[:,0]))


In [0]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
SCORING = "balanced_accuracy"  # faster + stable, swap to "f1_macro" once you see progress

# Logistic Regression (small grid)
lr = LogisticRegression(max_iter=2000)

lr_param_grid = [
    {
        "solver": ["liblinear"],
        "penalty": ["l2"],
        "C": [0.1, 1, 10],
        "class_weight": ["balanced"],
    },
    {
        "solver": ["liblinear"],
        "penalty": ["l1"],
        "C": [0.1, 1, 10],
        "class_weight": ["balanced"],
    },
]

lr_search = GridSearchCV(
    lr,
    lr_param_grid,
    scoring=SCORING,
    cv=cv,
    n_jobs=4,      # don’t always use -1 on shared clusters
    verbose=2,
)
lr_search.fit(X_train, y_train)
best_lr = lr_search.best_estimator_
print("Best LR params:", lr_search.best_params_)
print("Best LR CV score:", lr_search.best_score_)

# Random Forest (small grid)
rf = RandomForestClassifier(random_state=42, n_jobs=4)

rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [10, 20],
    "min_samples_leaf": [5, 10],
    "max_features": ["sqrt"],
    "class_weight": ["balanced_subsample"],
}

rf_search = GridSearchCV(
    rf,
    rf_param_grid,
    scoring=SCORING,
    cv=cv,
    n_jobs=4,
    verbose=2,
)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_
print("Best RF params:", rf_search.best_params_)
print("Best RF CV score:", rf_search.best_score_)


In [0]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression

cv2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = "balanced_accuracy"

lr2 = LogisticRegression(max_iter=8000)

lr2_param_grid = [{
    "solver": ["liblinear"],
    "penalty": ["l2"],
    "C": [0.2, 0.5, 0.8, 1, 1.5, 2, 3, 5, 10],
    "class_weight": ["balanced"],
}]

lr2_search = GridSearchCV(lr2, lr2_param_grid, scoring=SCORING, cv=cv2, n_jobs=4, verbose=2)
lr2_search.fit(X_train, y_train)

best_lr2 = lr2_search.best_estimator_
print("Best LR2 params:", lr2_search.best_params_)
print("Best LR2 CV score:", lr2_search.best_score_)


In [0]:
import numpy as np

classes = list(best_lr2.classes_)
print("classes:", classes)

pos_label = "no_step"
pos_idx = classes.index(pos_label)
proba_pos = best_lr2.predict_proba(X_test)[:, pos_idx]

print("no_step proba min/mean/max:", proba_pos.min(), proba_pos.mean(), proba_pos.max())
print("percentiles:", np.percentile(proba_pos, [1,5,10,25,50,75,90,95,99]))


In [0]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

pos_label = "no_step"
neg_label = "step"
classes = list(best_lr2.classes_)
pos_idx = classes.index(pos_label)
p = best_lr2.predict_proba(X_test)[:, pos_idx]

best = None
for t in np.linspace(0.35, 0.65, 61):
    pred = np.where(p >= t, pos_label, neg_label)
    bal = balanced_accuracy_score(y_test, pred)
    if best is None or bal > best[0]:
        best = (bal, t)

print("Best balanced accuracy:", best[0], "at threshold:", best[1])
pred = np.where(p >= best[1], pos_label, neg_label)
print(confusion_matrix(y_test, pred, labels=[pos_label, neg_label]))


In [0]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

rng = np.random.default_rng(42)

idx_step = np.where(y_train == "step")[0]
idx_no   = np.where(y_train == "no_step")[0]

idx_step_down = rng.choice(idx_step, size=len(idx_no), replace=False)
idx_bal = np.concatenate([idx_no, idx_step_down])
rng.shuffle(idx_bal)

X_bal = X_train[idx_bal]
y_bal = y_train[idx_bal]

lr_bal = LogisticRegression(solver="liblinear", penalty="l2", C=1, max_iter=8000)
lr_bal.fit(X_bal, y_bal)

y_pred = lr_bal.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))
print(confusion_matrix(y_test, y_pred, labels=["no_step","step"]))


In [0]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score

classes = list(lr_bal.classes_)
pos_label = "no_step"
neg_label = "step"
pos_idx = classes.index(pos_label)

p = lr_bal.predict_proba(X_test)[:, pos_idx]

for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    pred = np.where(p >= t, pos_label, neg_label)
    print("\nTHRESHOLD =", t)
    print("Balanced acc:", balanced_accuracy_score(y_test, pred))
    print("F1 macro:", f1_score(y_test, pred, average="macro"))
    print(confusion_matrix(y_test, pred, labels=[pos_label, neg_label]))
    print(classification_report(y_test, pred, zero_division=0))


In [0]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

rng = np.random.default_rng(42)

idx_step = np.where(y_train == "step")[0]
idx_no   = np.where(y_train == "no_step")[0]

ratio = 5  # keep 5x as many step as no_step
step_keep = min(len(idx_step), ratio * len(idx_no))
idx_step_down = rng.choice(idx_step, size=step_keep, replace=False)

idx_bal = np.concatenate([idx_no, idx_step_down])
rng.shuffle(idx_bal)

X_bal = X_train[idx_bal]
y_bal = y_train[idx_bal]

lr = LogisticRegression(solver="liblinear", penalty="l2", C=1, max_iter=8000)
lr.fit(X_bal, y_bal)

y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))
print(confusion_matrix(y_test, y_pred, labels=["no_step","step"]))


These future tests were attempts to increase accuracy overall; instead I discovered that the signal we have on what is and isn't a step is very vague. The ML system can either trade accuracy on the Steps or the Non-steps, not have both. This points to the data itself not explaining clearly what is and what isn't a step. For this case, we will accept the low no_step accuracy for the high step accuracy, as a grand majority of the data is step data regardless.